In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
def cook(site_uid, recipe, target_col):
    # setup the data and main split
    master_df = recipe(site_uid)
    split = int(len(master_df) * 0.9)
    master_train = master_df.iloc[:split]
    master_eval = master_df.iloc[split:]
    
    # define the target and features
    target = target_col
    feature_cols = [c for c in master_df.columns if c not in (target, "site_uid", "date")]
    X, X_eval = master_train[feature_cols], master_eval[feature_cols]
    y, y_eval = master_train[target], master_eval[target]
    
    kfold = TimeSeriesSplit(n_splits = 10,
                            test_size = 90)

    def report(name, Xs, ys):
        p = model.predict(Xs)
        rmse = np.sqrt(mean_squared_error(ys, p))
        mae  = mean_absolute_error(ys, p)
        r2   = r2_score(ys, p)
        print(f"  {name:5s} n={len(ys):4d}  RMSE={rmse:6.3f}  MAE={mae:6.3f}  R2={r2:7.3f}  (y std={ys.std():.3f})")

    for fold, (train_index, test_index) in enumerate(kfold.split(master_train)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        model = xgb.XGBRegressor(
            n_estimators=3000,      # ceiling; early stopping picks the real count
            learning_rate=0.02,
            max_depth=3,
            min_child_weight=10,
            subsample=0.7,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            random_state=42,
            eval_metric="rmse",
            early_stopping_rounds=50,
        )

        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        print(f"Iteration {fold} with train/test split {X_train.shape[0]}/{X_test.shape[0]}")
        print("  best #trees (chosen by early stopping):", model.best_iteration + 1)
        
        preds = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)
        
        report("train", X_train,  y_train)
        report("val",   X_test, y_test)
        report("test",  X_eval,  y_eval)

    # Review the top shifts in global feature importance
    importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nTop 15 Enhanced Feature Importances:\n", importances.head(15))


In [ ]:
import sys
sys.path.insert(0, "../")

from data import get_data
from data.recipes import preet_df1
from data.features import (
    agg_crops, agg_surplus, agg_weather_w_lag, daily_nitrate, lagged_sensor_nitrate,
    nitrate_rolling, nitrate_avg_seasonal, nitrate_avg_calendar, doy_climatology_pure_signal,
)
from data.transforms import flatten_buckets, merge_on_date, match_seasonal

NEIGHBORS = ["USGS-05482500", "USGS-05484500", "USGS-05465500"]
EDGES = [50_000, 150_000]
VEL = 0.8
TARGETCOL = "nitrate_con"

def _covariates(site):
    """weather (travel-time lagged buckets) + crops/surplus (exp-decay buckets) + pure calendar."""
    wb = flatten_buckets(agg_weather_w_lag(site, edges=EDGES, water_velocity=VEL))
    cb = flatten_buckets(agg_crops(site, edges=EDGES, lam=10_000, exp=True))
    sb = flatten_buckets(agg_surplus(site, edges=EDGES, lam=10_000, exp=True))
    n_daily = daily_nitrate(site).rename(TARGETCOL)
    doy = doy_climatology_pure_signal(n_daily)  # doy_sin/doy_cos
    return n_daily, [wb, cb, sb, doy]

def recipe_A(site):
    """Covariates only: weather + land-use + pure calendar. No nitrate-derived features."""
    n_daily, parts = _covariates(site)
    out = merge_on_date([n_daily, *parts], spine=n_daily.index)
    return out.dropna(subset=[TARGETCOL]).reset_index(drop=True)

def recipe_B(site):
    """A + the site's OWN past nitrate (autoregressive; sensor sees its own history)."""
    n_daily, parts = _covariates(site)
    own = [lagged_sensor_nitrate([site], shift=k) for k in (1, 2, 3, 7, 14, 30)]
    return merge_on_date([n_daily, *parts, *own], spine=n_daily.index)

def recipe_C(site):
    """A + cross-site climatology (causal) + neighbour sensors' past nitrate."""
    n_daily, parts = _covariates(site)
    dates = n_daily.index
    clim = [
        nitrate_rolling("7D", center=False).rename("nroll_7"),
        nitrate_rolling("30D", center=False).rename("nroll_30"),
        nitrate_avg_calendar("D").rename("ncal_d"),
        match_seasonal(dates, nitrate_avg_seasonal("D")).rename("ndoy"),
        match_seasonal(dates, nitrate_avg_seasonal("W")).rename("nwoy"),
        match_seasonal(dates, nitrate_avg_seasonal("M")).rename("nmoy"),
    ]
    neigh = [lagged_sensor_nitrate(NEIGHBORS, shift=k) for k in (1, 3, 7)]
    return merge_on_date([n_daily, *parts, *clim, *neigh], spine=dates)

In [ ]:
SITE = "USGS-05482300"
TARGETCOL = "nitrate_con"

cook(SITE, recipe_A, TARGETCOL)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def model_clf(site_uid, recipe, threshold=10, eval_metric="logloss", balance=False):
    df = recipe(site_uid)
    target = "nitrate_con"
    feat = [c for c in df.columns if c not in (target, "site_uid", "date")]
    X = df[feat]
    y = (df[target] >= threshold).astype(int)          # 1 = violation (>= threshold)

    n = len(df); i_tr, i_val = int(n*0.70), int(n*0.85)
    Xtr, ytr = X.iloc[:i_tr], y.iloc[:i_tr]
    Xva, yva = X.iloc[i_tr:i_val], y.iloc[i_tr:i_val]   # validation -> early stopping
    Xte, yte = X.iloc[i_val:], y.iloc[i_val:]           # test -> report only

    # only up-weight positives if they're the MINORITY; otherwise it miscalibrates
    spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1) if balance else 1.0
    clf = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.02, max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7, reg_lambda=5.0, random_state=42,
        objective="binary:logistic", eval_metric=eval_metric,
        early_stopping_rounds=50, scale_pos_weight=spw,
    )
    clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    print(f"best #trees = {clf.best_iteration + 1}  (threshold={threshold}, balance={balance})")

    def report(name, Xs, ys):
        p = clf.predict_proba(Xs)[:, 1]                # P(violation) for each day
        if ys.nunique() < 2:
            print(f"{name:5s} n={len(ys):4d} base={ys.mean():.3f}  (one class only)"); return
        print(f"{name:5s} n={len(ys):4d}  base={ys.mean():.3f}  meanP={p.mean():.3f}  "
              f"AUC={roc_auc_score(ys,p):.3f}  PR-AUC={average_precision_score(ys,p):.3f}  "
              f"Brier={brier_score_loss(ys,p):.3f}")

    report("train", Xtr, ytr)
    report("val",   Xva, yva)
    report("test",  Xte, yte)
    imp = pd.Series(clf.feature_importances_, index=feat).sort_values(ascending=False)
    print("\nTop 12 features:\n", imp.head(12).to_string())
    return clf

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

def split_proba(clf, site_uid, recipe, threshold=10, part="test"):
    """P(violation) and true labels for one chronological split (matches model_clf)."""
    df = recipe(site_uid)
    feat = [c for c in df.columns if c not in ("nitrate_con", "site_uid", "date")]
    y = (df["nitrate_con"] >= threshold).astype(int)
    n = len(df); i_tr, i_val = int(n*0.70), int(n*0.85)
    sl = {"train": slice(0, i_tr), "val": slice(i_tr, i_val), "test": slice(i_val, n)}[part]
    return y.iloc[sl].to_numpy(), clf.predict_proba(df[feat].iloc[sl])[:, 1]

def threshold_sweep(y_true, proba, thresholds=None):
    """Precision / recall / F1 / false-positive-rate vs decision cutoff. Returns a table."""
    y_true, proba = np.asarray(y_true), np.asarray(proba)
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum()); fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum()); tn = int(((pred == 0) & (y_true == 0)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        fpr  = fp / (fp + tn) if fp + tn else 0.0
        rows.append(dict(thr=t, precision=prec, recall=rec, f1=f1, fpr=fpr, flagged=tp + fp))
    tab = pd.DataFrame(rows)
    best = tab.loc[tab.f1.idxmax()]
    print(tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    print(f"\nbest F1 @ thr={best.thr:.2f}:  precision={best.precision:.3f}  recall={best.recall:.3f}  f1={best.f1:.3f}")
    return tab

def reliability_curve(y_true, proba, n_bins=10, title="Reliability"):
    """Calibration curve (observed freq vs predicted prob) + histogram of predictions."""
    y_true, proba = np.asarray(y_true), np.asarray(proba)
    frac_pos, mean_pred = calibration_curve(y_true, proba, n_bins=n_bins, strategy="quantile")
    brier = brier_score_loss(y_true, proba)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 6), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 1]})
    ax1.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
    ax1.plot(mean_pred, frac_pos, "o-", label=f"model (Brier={brier:.3f})")
    ax1.axhline(y_true.mean(), color="gray", ls=":", lw=1, label=f"base rate={y_true.mean():.2f}")
    ax1.set_ylabel("observed frequency"); ax1.set_ylim(0, 1)
    ax1.legend(loc="best"); ax1.set_title(title)
    ax2.hist(proba, bins=20, range=(0, 1), color="steelblue")
    ax2.set_xlabel("predicted P(violation)"); ax2.set_ylabel("count")
    plt.tight_layout(); plt.show()
    return brier
